# Does Income Inequality Predict Higher Diabetes Prevalence in U.S. Counties?

### 2018-2025 County-Level Analysis, Controlling for Education Attainment and Food Access

**Author**: Shunji Lewandowski (*Economics and Human Health Joint Major*)  
**Date**: May 2, 2025  
**Course**: ECON 220 Lab - Emory University

## Abstract

This study investigates the relationship between median household income and diabetes prevalence across U.S. counties from 2018 to 2025, controlling for high school educational attainment and food environment quality. Using publicly available county-level data from the County Health Rankings & Roadmaps (CHR), the analysis applies correlation and OLS regression to examine how socioeconomic factors shape health disparities. Results show a statistically significant inverse relationship between median income and diabetes rates across all years studied (Pearson r ranging from -0.47 to -0.69, p < 0.001). Even after accounting for high school completion rates and food access, the relationship remains consistent. These findings underscore the role of structural economic inequality in driving chronic disease prevalence and support calls for preventative, equity-focused health policy.

## 1. Introduction

### Background and Context

Diabetes mellitus -- particularly type 2 diabetes -- remains a pressing public health challenge in the United States. According to the CDC (2021), an estimated 11.6% of the adult population has diabetes, costing the nation approximately $413 billion in direct and indirect expenses in 2021 alone. Type 2 diabetes is largely preventable and influenced by modifiable risk factors, yet its prevalence continues to climb, disproportionately affecting low-income and minority communities.

While much focus has been placed on individual behaviors such as diet and exercise, a growing body of evidence highlights the role of structural factors in shaping health outcomes. Income inequality has been linked to higher chronic disease rates, including diabetes (NASEM, 2019). Individuals in lower-income areas are more likely to live in "food deserts" -- geographic areas with limited access to affordable and nutritious food -- and are less likely to access timely preventive care. These disadvantages can be further exacerbated by lower educational attainment, which correlates with limited health literacy and reduced agency in managing chronic disease risks.

### Purpose

This study quantifies how strongly median income levels predict diabetes prevalence while isolating the effects of educational attainment and food environment quality. By leveraging longitudinal county-level data and controlling for confounding variables, this research aims to generate actionable insights for public health policymakers. Unlike clinical research that focuses on treatment, this study targets the upstream socioeconomic conditions that contribute to disease, emphasizing the importance of prevention.

### Research Question

**Does median household income predict diabetes prevalence at the county level, after controlling for educational attainment and food environment quality?**

## 2. Methodology

### Data Source

Data were sourced from the [County Health Rankings & Roadmaps (CHR)](https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation), a nationally recognized public database compiled by the Robert Wood Johnson Foundation and the University of Wisconsin Population Health Institute. The CHR aggregates health-related metrics from federal sources including the CDC, Census Bureau, and USDA. Annual datasets from 2018 to 2025 were compiled, encompassing 2,500 to 3,000+ counties per year.

### Variables

| Variable | Description | Measurement |
|---|---|---|
|  | % of adults aged 18+ with diagnosed diabetes | Continuous (0-100%) |
|  | Median household income per county | Continuous (USD) |
|  | % of adults aged 25+ with high school diploma | Continuous (0-100%) |
|  | Access to healthy food (0 = worst, 10 = best) | Continuous |

- **Timeframe**: 2018-2025 (8 years)
- **Scope**: All U.S. counties with available data (~2,500-3,078 per year)
- **Total Observations**: ~21,000+ county-year combinations

## 3. Setup and Data Preparation

### 3.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully.")

### 3.2 Load and Combine Data

Each year's CSV uses slightly different column naming conventions (e.g., title case vs. sentence case). The rename map standardizes all variants to consistent variable names.

In [ ]:
# File paths (relative to notebook location)
base_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()

csv_files = {
    2018: "analytic_data2018_0.csv",
    2019: "analytic_data2019.csv",
    2020: "analytic_data2020_0.csv",
    2021: "analytic_data2021.csv",
    2022: "analytic_data2022.csv",
    2023: "analytic_data2023_0.csv",
    2024: "analytic_data2024.csv",
    2025: "analytic_data2025.csv",
}

# Map all column name variants across years to standardized names
rename_map = {
    "Median Household Income raw value": "median_income",
    "Median household income raw value": "median_income",
    "Diabetes Prevalence raw value": "diabetes_rate",
    "Diabetes prevalence raw value": "diabetes_rate",
    "High School Graduation raw value": "hs_graduation_rate",
    "High school graduation raw value": "hs_graduation_rate",
    "High School Completion raw value": "hs_graduation_rate",
    "High school completion raw value": "hs_graduation_rate",
    "Food Environment Index raw value": "food_env_index",
    "Food environment index raw value": "food_env_index",
}

columns_to_keep = [
    "State Abbreviation", "Name", "5-digit FIPS Code", "Release Year"
] + list(rename_map.keys())

# Load and combine all years
frames = []
for year, filename in csv_files.items():
    path = os.path.join(base_dir, filename)
    df = pd.read_csv(path, usecols=lambda col: col in columns_to_keep, low_memory=False)
    df["Year"] = year
    df = df.rename(columns=rename_map)
    # Some years have both "Completion" and "Graduation" columns that map to
    # the same name — keep only the first occurrence to avoid duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]
    frames.append(df)

combined_df = pd.concat(frames, ignore_index=True)
print(f"Combined dataset: {combined_df.shape[0]:,} rows x {combined_df.shape[1]} columns")
print(f"Years covered: {sorted(combined_df['Year'].unique())}")

### 3.3 Clean Data

Convert columns to numeric types and drop rows with missing values in any of the four key variables.

In [ ]:
analysis_cols = ["median_income", "diabetes_rate", "hs_graduation_rate", "food_env_index"]

# Convert to numeric (some values may be stored as strings)
combined_df[analysis_cols] = combined_df[analysis_cols].apply(pd.to_numeric, errors="coerce")

# Drop rows with missing values in key columns
n_before = len(combined_df)
combined_df = combined_df.dropna(subset=analysis_cols)
n_after = len(combined_df)

print(f"Rows before cleaning: {n_before:,}")
print(f"Rows after cleaning:  {n_after:,}")
print(f"Rows dropped:         {n_before - n_after:,}")
print(f"
Descriptive statistics:")
combined_df[analysis_cols].describe().round(2)

### 3.4 Outlier Removal (IQR Method)

To reduce the influence of extreme values, we remove observations outside 1.5x the interquartile range for median income and diabetes rate.

In [ ]:
def remove_outliers_iqr(df, cols):
    """Remove rows where any column value falls outside 1.5 * IQR."""
    mask = pd.Series(True, index=df.index)
    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        mask &= df[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return df[mask].copy()

cleaned_df = remove_outliers_iqr(combined_df, ["median_income", "diabetes_rate"])

print(f"Before outlier removal: {len(combined_df):,} rows")
print(f"After outlier removal:  {len(cleaned_df):,} rows")
print(f"Outliers removed:       {len(combined_df) - len(cleaned_df):,} rows")

## 4. Exploratory Data Analysis

### 4.1 Year-by-Year Scatterplots with Regression Lines

Each panel shows the bivariate relationship between median household income and diabetes prevalence for a single year, with an OLS regression line overlaid.

In [ ]:
years = sorted(cleaned_df["Year"].unique())
palette = sns.color_palette("tab10", n_colors=len(years))
year_colors = dict(zip(years, palette))

fig, axes = plt.subplots(2, 4, figsize=(20, 10), sharex=True, sharey=True)
axes = axes.flatten()

for i, year in enumerate(years):
    ax = axes[i]
    df_year = cleaned_df[cleaned_df["Year"] == year]

    sns.scatterplot(
        data=df_year, x="median_income", y="diabetes_rate",
        color=year_colors[year], alpha=0.3, s=15, ax=ax
    )

    # Fit and plot regression line
    model = smf.ols("diabetes_rate ~ median_income", data=df_year).fit()
    x_range = np.linspace(df_year["median_income"].min(), df_year["median_income"].max(), 100)
    ax.plot(x_range, model.predict(pd.DataFrame({"median_income": x_range})),
            color="black", linestyle="--", linewidth=1.5)

    ax.set_title(f"{year}  (r = {df_year['median_income'].corr(df_year['diabetes_rate']):.3f})",
                 fontsize=11)
    ax.set_xlabel("Median Income ($)" if i >= 4 else "")
    ax.set_ylabel("Diabetes Rate (%)" if i % 4 == 0 else "")

fig.suptitle("Diabetes Prevalence vs. Median Household Income by Year (Cleaned)",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4.2 Combined Scatterplot (All Years)

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=cleaned_df, x="median_income", y="diabetes_rate",
    hue="Year", palette="tab10", alpha=0.3, s=15
)

# Combined regression line
model_combined = smf.ols("diabetes_rate ~ median_income", data=cleaned_df).fit()
x_range = np.linspace(cleaned_df["median_income"].min(), cleaned_df["median_income"].max(), 100)
y_pred = model_combined.predict(pd.DataFrame({"median_income": x_range}))
plt.plot(x_range, y_pred, color="black", linestyle="--", linewidth=2)

slope = model_combined.params["median_income"]
intercept = model_combined.params["Intercept"]
r2 = model_combined.rsquared
plt.text(
    0.02, 0.98,
    f"y = {intercept:.4f} + ({slope:.8f}) * income
R² = {r2:.3f}",
    transform=plt.gca().transAxes, fontsize=10,
    verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

plt.title("Diabetes Prevalence vs. Median Household Income (2018-2025, Cleaned)",
          fontsize=13, fontweight="bold")
plt.xlabel("Median Household Income ($)")
plt.ylabel("Diabetes Prevalence Rate (%)")
plt.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 5. Statistical Analysis

### 5.1 Pearson Correlation (Year-by-Year)

In [ ]:
print("Pearson Correlation: Median Income vs. Diabetes Rate
")
print(f"{'Year':<8} {'n':>6} {'Pearson r':>12} {'p-value':>12}")
print("-" * 40)

for year in years:
    df_year = cleaned_df[cleaned_df["Year"] == year]
    r, p = stats.pearsonr(df_year["median_income"], df_year["diabetes_rate"])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"{year:<8} {len(df_year):>6,} {r:>12.3f} {p:>12.2e} {sig}")

# Combined
r_all, p_all = stats.pearsonr(cleaned_df["median_income"], cleaned_df["diabetes_rate"])
print("-" * 40)
print(f"{'All':<8} {len(cleaned_df):>6,} {r_all:>12.3f} {p_all:>12.2e} ***")
print("
*** p < 0.001, ** p < 0.01, * p < 0.05")

### 5.2 Multiple Linear Regression (Year-by-Year)

For each year, we fit an OLS model:



This controls for education and food access when estimating the effect of income on diabetes.

In [ ]:
formula = "diabetes_rate ~ median_income + hs_graduation_rate + food_env_index"

print("Multiple Linear Regression Results (Year-by-Year)
")
print(f"{'Year':<6} {'n':>6} {'R²':>8} {'Adj R²':>8} {'β(income)':>14} {'p(income)':>12} {'β(educ)':>12} {'β(food)':>12}")
print("-" * 82)

for year in years:
    df_year = cleaned_df[cleaned_df["Year"] == year]
    model = smf.ols(formula, data=df_year).fit()
    print(
        f"{year:<6} {len(df_year):>6,} {model.rsquared:>8.3f} {model.rsquared_adj:>8.3f} "
        f"{model.params['median_income']:>14.8f} {model.pvalues['median_income']:>12.2e} "
        f"{model.params['hs_graduation_rate']:>12.4f} {model.params['food_env_index']:>12.4f}"
    )

### 5.3 Combined Regression Model (All Years)

In [ ]:
# Full model with all years
model_full = smf.ols(formula, data=cleaned_df).fit()
print(model_full.summary())

### 5.4 Hypothesis Test on Income Coefficient

**H0**: The coefficient on median income = 0 (income does not predict diabetes)  
**H1**: The coefficient on median income ≠ 0 (income predicts diabetes)

In [ ]:
slope = model_full.params["median_income"]
se = model_full.bse["median_income"]
t_stat = model_full.tvalues["median_income"]
p_val = model_full.pvalues["median_income"]
df_resid = int(model_full.df_resid)

print("T-Test on Slope: Median Income as Predictor of Diabetes Rate")
print("=" * 55)
print(f"Coefficient (β):    {slope:.10f}")
print(f"Standard Error:     {se:.10f}")
print(f"t-statistic:        {t_stat:.4f}")
print(f"p-value:            {p_val:.2e}")
print(f"Degrees of Freedom: {df_resid:,}")
print()

# Practical interpretation: effect per $10,000 increase
# diabetes_rate is a proportion (e.g., 0.11 = 11%), so multiply by 100 for pp
effect_per_10k = slope * 10_000 * 100
print(f"Interpretation: A $10,000 increase in median household income")
print(f"is associated with a {abs(effect_per_10k):.2f} percentage point")
print(f"{'decrease' if slope < 0 else 'increase'} in diabetes prevalence.")
print()

alpha = 0.05
if p_val < alpha:
    print(f"Result: REJECT H0 at α = {alpha}. Median income is a statistically")
    print(f"significant predictor of diabetes prevalence (p < 0.001).")
else:
    print(f"Result: FAIL TO REJECT H0 at α = {alpha}.")

## 6. Discussion

The inverse relationship between median household income and diabetes prevalence observed in this study aligns with well-documented socioeconomic health disparities in the United States (CDC, 2021). Lower-income communities face systemic barriers that exacerbate diabetes risk, including limited access to nutritious food and preventive healthcare. Healthier food options often cost 2-3x more per calorie than processed alternatives, which forces many low-income households to rely on calorie-dense, nutrient-poor diets that contribute to obesity and insulin resistance (NASEM, 2019).

Key findings:

1. **Consistent negative correlation**: Pearson correlations ranged from approximately -0.42 to -0.67 across all years, all statistically significant at p < 0.001.

2. **Robust to controls**: The relationship between income and diabetes persists even after controlling for education and food environment, suggesting income captures additional pathways to health beyond these two factors.

3. **COVID-19 impact**: The weakest correlations appeared in 2020-2021 (r ≈ -0.42), consistent with pandemic-related disruptions in healthcare access and diabetes diagnosis.

4. **Practical significance**: Each $10,000 increase in median household income is associated with approximately a 0.86 percentage point decrease in county-level diabetes prevalence.

### Policy Implications

- Expanding economic interventions (EITC, SNAP) to improve food security
- Mandating insurance coverage for evidence-based prediabetes prevention programs (e.g., the CDC's National Diabetes Prevention Program), which can reduce progression to diabetes by up to 58% (CDC, 2021)
- Subsidizing healthy food retailers in food deserts and regulating predatory marketing of sugar-sweetened beverages

### Limitations

- Ecological study design (county-level aggregates; cannot infer individual-level causation)
- Potential confounders not included (e.g., race/ethnicity composition, urbanization, healthcare access)
- Cross-sectional analysis within each year; does not establish temporal causation
- Column naming inconsistencies across CHR data releases required careful harmonization

## 7. References

- Attia, P. (2023). *Outlive: The Science and Art of Longevity*. Harmony Books.
- Centers for Disease Control and Prevention. (2021). National Diabetes Statistics Report. https://www.cdc.gov/diabetes/php/data-research/index.html
- County Health Rankings & Roadmaps. (2024). Data Documentation & Sources. University of Wisconsin Population Health Institute. https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation
- National Academies of Sciences, Engineering, and Medicine. (2019). The Role of Social Determinants in Diabetes. In *Diabetes in America* (3rd ed.). https://www.ncbi.nlm.nih.gov/books/NBK425845/
- U.S. Department of Health and Human Services. (2022). Building the Evidence Base for Social Determinants of Health Interventions. https://aspe.hhs.gov/reports/building-evidence-base-social-determinants-health-interventions